# Domain 4: Eval, Testing, and Debugging
**Exam**: Claude Certified Developer – Foundations (CCDV-F)
**Weighting**: 2.6% of exam
**Depth**: deep

## Domain Overview

This domain covers debugging and error handling techniques: error type identification,
recovery strategy selection, trace analysis to identify failure modes, and isolating whether a
problem originates in the integration layer or the model's output.

**Why it matters**: even a small-weight domain, this is a core daily skill — developers must be able
to quickly determine whether a bug is "their code" or "the model" and respond accordingly.

**Key concepts to know**: error taxonomy, recovery strategy selection, trace analysis, and
integration-layer vs. model-output isolation.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Error type identification and recovery strategy selection

Errors generally fall into: transport/infra (timeouts, rate limits - retryable), request
validation (malformed input - not retryable without a fix), and model-output issues (wrong format,
hallucination - needs prompt/schema fixes, not a blind retry).

In [ ]:
def classify_error(error_message: str) -> str:
    msg = error_message.lower()
    if "429" in msg or "rate limit" in msg or "timeout" in msg:
        return "transport/infra - retryable with backoff"
    if "invalid request" in msg or "400" in msg:
        return "request validation - fix the request, do not blindly retry"
    if "unexpected format" in msg or "schema" in msg:
        return "model-output issue - fix prompt/schema, retry with feedback"
    return "unknown - needs investigation"

for e in ["429 Too Many Requests", "400 Invalid Request: missing field", "Response did not match schema"]:
    print(e, "->", classify_error(e))

### Trace analysis to identify failure modes

A trace (the full sequence of requests, tool calls, and responses) reveals whether a
failure happened at request construction, tool execution, or the model's final synthesis. Reading a
trace top-to-bottom, rather than only looking at the final error, is often necessary to find the true
point of failure.

In [ ]:
trace = [
    {"step": "build_request", "status": "ok"},
    {"step": "call_tool:lookup_order", "status": "ok", "result": {"order_id": "O1", "amount": None}},
    {"step": "model_synthesis", "status": "error", "message": "Cannot format refund amount: None"},
]
def find_failure_point(trace: list[dict]) -> dict:
    for step in trace:
        if step["status"] == "error":
            return step
    return {"step": "none", "status": "ok"}

print(find_failure_point(trace))
print("Root cause is upstream: lookup_order returned amount=None, not a synthesis bug.")

### Isolating integration-layer vs. model-output problems

A quick isolation test: replay the exact same model output through the integration
layer's parsing/handling code in isolation. If it fails the same way, the bug is in the integration
layer. If it succeeds, the bug is in what the model produced (or in how the request was built).

In [ ]:
def isolate_bug(model_output: str, integration_parser) -> str:
    try:
        integration_parser(model_output)
        return "model output was fine - check request construction/prompt instead"
    except Exception as e:
        return f"integration layer bug reproduced: {e}"

def buggy_parser(text: str):
    return int(text)  # BUG: assumes output is always a plain integer string

print(isolate_bug("42", buggy_parser))
print(isolate_bug("The answer is 42", buggy_parser))

## Mini Hands-On Project

**Covers**: all Domain 4 sub-objectives.

**Scenario**: A production alert reports "refund processing failing intermittently." Diagnose it.

### Step 1: Classify the reported error

In [ ]:
print(classify_error("Response did not match schema"))

### Step 2: Analyze the trace to find the failure point

In [ ]:
print(find_failure_point(trace))

### Step 3: Isolate whether it's an integration or model-output bug

In [ ]:
print(isolate_bug("42", buggy_parser))

## Lab Exercises

### Exercise 1: Error classification batch

**Objective**: Classify 5 new error messages.

**Produce**: 5 classifications.

In [ ]:
errors = ["503 Service Unavailable", "400 Bad Request: unknown field 'foo'",
          "Extraction returned a string where a number was expected", "Connection timed out after 30s",
          "Tool call failed: permission denied"]
# TODO: call classify_error (extend it if needed for the permission-denied case) for each
for e in errors:
    pass

# Expected output: Correct classification for the first 4; recognizes the 5th needs a new 'permission' category not currently handled.

In [ ]:
# STRETCH GOAL: extend classify_error to add a "permission" category for permission-denied errors.
def classify_error_v2(error_message: str) -> str:
    pass
print(classify_error_v2("Tool call failed: permission denied"))

# Expected output: A new, correctly-triggered 'permission' category distinct from the other three.

### Exercise 2: Multi-step trace debugging

**Objective**: Find the failure point in a longer 5-step trace.

**Produce**: The identified failure step and root cause explanation.

In [ ]:
longer_trace = [
    {"step": "build_request", "status": "ok"},
    {"step": "call_tool:get_customer", "status": "ok", "result": {"id": "C1"}},
    {"step": "call_tool:lookup_order", "status": "ok", "result": {"order_id": "O9", "amount": 75.0}},
    {"step": "call_tool:process_refund", "status": "error", "message": "amount exceeds single-transaction limit"},
    {"step": "model_synthesis", "status": "not_reached"},
]
# TODO: use find_failure_point (or similar logic) to locate and explain the failure
failure = None  # TODO
print(failure)

# Expected output: Correctly identifies the process_refund step as the failure point, with the transaction-limit message as root cause.

In [ ]:
# STRETCH GOAL: propose a recovery strategy specific to this root cause (not a generic retry).
recovery_strategy = "..."  # TODO

# Expected output: A specific recovery strategy, e.g. splitting into multiple smaller transactions or escalating to a human, not a blind retry.

### Exercise 3: Integration vs. model isolation test

**Objective**: Write an isolation test for a JSON-parsing integration bug.

**Produce**: A test proving the bug is in the parser, not the model output.

In [ ]:
def json_parser(text: str):
    import json
    return json.loads(text)

well_formed_output = '{"status": "ok", "amount": 50}'
malformed_output = '{status: ok, amount: 50}'  # model occasionally omits quotes around keys

# TODO: use isolate_bug-style logic to determine: is this a model-output problem or an integration-layer problem?
result_well_formed = None  # TODO
result_malformed = None  # TODO
print(result_well_formed, result_malformed)

# Expected output: Well-formed output parses fine (parser is OK); malformed output fails, but the root fix should target model output consistency (e.g., use tool_use for guaranteed JSON) rather than just patching the parser.

In [ ]:
# STRETCH GOAL: propose the most robust fix (parser patch vs. switching to tool_use/JSON schema
# enforcement) and justify it.
robust_fix = "..."  # TODO

# Expected output: A recommendation favoring tool_use/schema enforcement over ad-hoc parser patching, with justification.

## Self-Assessment: Eval, Testing, and Debugging

Answer each question, then check the Answers cell below.


**Q1.** What are the three broad error categories discussed, and which is safe to retry automatically?

**Q2.** Why is reading a full execution trace top-to-bottom often necessary rather than just looking at the final error?

**Q3.** What is the quickest way to determine whether a bug is in the integration layer or in the model's output?

**Q4.** Why is a blind retry an inappropriate recovery strategy for a 'business rule violation' error?

**Q5.** Give an example of a model-output problem that a schema/tool_use fix would resolve better than a parser patch.

## Answers


**A1.** Transport/infra, request validation, and model-output issues; transport/infra errors (timeouts, rate limits) are generally safe to retry with backoff.

**A2.** Because the final error is often a downstream symptom; the true root cause (e.g., a bad upstream tool result) may appear earlier in the trace.

**A3.** Replay the exact same model output through the integration/parsing code in isolation - if it fails identically, the bug is in the integration layer.

**A4.** Because retrying won't change the outcome; the request will fail the same business rule again until the underlying request or logic is fixed.

**A5.** Inconsistently-formatted JSON output (e.g., missing quotes around keys) - enforcing schema via tool_use eliminates this class of error at the source.

---
**Next**: [Domain 5: Model Selection and Optimization](./05-model-selection-and-optimization.ipynb)
**Back**: [README](./README.md)